# Chirp+Cell Typing-dev

contact: ron.w.ditullio@gmail.com based on Guilhelm's guilhelm-dev branch

execution time:

Tested on Ubuntu 24.04.2 LTS (32 cores, 188 GiB RAM, Intel(R) Core(TM) i9-14900K)


In [ ]:
%load_ext autoreload
%autoreload 2

# import packages: primary
import os
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import loadmat
import csv

# import packages for clustering #2026-01-21 RWD: sklearn is not in the environment by default.  Maybe add to yaml?
from scipy import interpolate

# import custom packages
import params
import chirp_and_celltyping_ron as analysis
import utils
import gc

## Cell 1: Load triggers and spikes + select chirp type !!!!

In [ ]:
old = False  # 2026-01-22 RWD: for now have old set outside function and returned by function.  Change to default inside if this is rarely toggeled.

cells, spike_times, spike_trains, stim_onsets, check_directory, CT_directory, old = (
    analysis.get_all_inputs_for_chirp_analysis(params, old)
)

## Cell 2: Chirp rasters

In [ ]:
cell_data = analysis.compute_chirp_rasters(
    cells, spike_times, stim_onsets, old=False, n_bins=800, n_bins_small=16000
)

## Cell 3: Plot Chirp rasters

With added spatial STA in order to be able to use this plot alone to do cell selection for clustering

#### <center><i>REQUIRES CELL 2 RUN AND CHECKERBOARD ANALYSIS</center>

In [ ]:
analysis.plot_chirp_rasters(
    cells, cell_data, CT_directory, check_directory
)  # 2026-01-22 RWD code should work but don't have needed vec file
# specifically: Euler_50Hz_20reps_1024x768pix.vec not found.

## Cell 4: Select the cells suited for clustering and save them
Those that have a nice response to the chirp and a well defined STA

2026-01-22 RWD: If no file and cells are not entered manually, will go through all cells.  Takes a bit.

In [ ]:
## 2026-01-22 RWD for now just leave these in the cell to be edited.  Could change to input but likely easier to just leave alone.


# Manually input here the cells ids to use for the clustering.
# If empty, checks if selection has already been done, otherwise, starts selection

# 2026-01-22 RWD just picked random cells for testing out the edits.
good_sta_cells = [1, 2, 15]  # Based on STA
good_chirp_cells = [1, 2, 15]  # Based on chirp

selected_cells, selected_cells_sta, selected_cells_chirp = (
    analysis.select_and_save_cells_for_clustering(
        cells, good_sta_cells, good_chirp_cells, CT_directory, params
    )
)

#### Modify manually your selection

In [ ]:
# 2026-01-22 same thing as above.  Let users just do this in the notebook for now.

# manually add or remove cell numbers if you failed
selected_cells_sta_to_add = []
selected_cells_sta_to_remove = []

selected_cells_chirp_to_add = []
selected_cells_chirp_to_remove = []

remove_any_way = []

analysis.modify_cells_for_clustering(
    selected_cells_sta,
    selected_cells_sta_to_add,
    selected_cells_sta_to_remove,
    selected_cells_chirp,
    selected_cells_chirp_to_add,
    selected_cells_chirp_to_remove,
    remove_any_way,
    CT_directory,
    params,
)

## Cell 5: Cell typing: Agglomerative Clustering

-you want to move the distance threshold until you have roughly 50 clusters <br>
-you want to have a number of PCs that cumulatively can explain around 80% of the variance in the dataset <br>
-you can decide how many components of the STA to choose in the clustering (usually 2 if the checkerboard recording is reliable and 1 otherwise)

#### <center><i>REQUIRES CELL 2, CELL 4 AND CELL 5 RUN </center>

In [ ]:
# 2026-01-22 RWD: May want to
# separate this out into two functions and two cells.  One runs the initial fit and one runs all other fits.
# only makes sense though if the model is passed between functions.  For now just have one function called run
# and leave it alone.

# change dist_thres to adapt the cut of the dendrogram and select the number of clusters
#####################################################################
dist_thres = 25  # 13
#####################################################################

# Select number of PCs to keep so to explain ~80% of the variance
#####################################################################
n_components_psth = 2  # 25#13#16 #2026-01-22 RWD: just setting to 2 for now for speedy check.  OG val was 25
#####################################################################

#####################################################################
n_components_sta_tc = 2
#####################################################################

sparse = False

psth_z, sta_results, model = analysis.run_cell_typing_AC(
    dist_thres,
    n_components_psth,
    n_components_sta_tc,
    cell_data,
    selected_cells,
    check_directory,
    sparse,
)

## Cell 6: Save each cell's cluster number in the chirp response dictionary

In [ ]:
cell_data = analysis.save_cluster_number_for_cells(selected_cells, cell_data, model)

# 2026-01-22 RWD: just one to check not assigned vs assigned

print(cell_data[1]["type"])

print(cell_data[0]["type"])

## Cell 7: Compute cross-correlation inside clusters

In [ ]:
# Number of bins to make psth
n_bins = 16000  # CHANGE HERE THE NUMBER OF BINS IF NECESSARY
max_shift = 50

# 2026-01-22 RWD: Note- variable "old" is set in Cell 1


# 2026-01-22 RWD: Selfnote- for this to run properly likely need multiple cells in the same cluster which means I have to
# actually and properly do clustering.  For the moment the code starts but runs into an issue with division that again likely comes from
# only having looked at 3 cells.  Skip for now and keep converting cells into functions to get to notebook 5.
# 2026-01-23 Clarifying note the code chunk runs we just get runtime warnings, likely due to above.
cell_data = analysis.compute_intracluster_crosscorr(
    cell_data, sta_results, n_bins, max_shift, old
)

## Cell 8: Create a summary figure for each cluster type

#### <center><i>REQUIRES CELL 6 RUN </center>

In [ ]:
# 2026-01-22 RWD: corrs key doesn't exist in cell data like because we didn't run above with actual clusters.  For now moving on
# last cell of this notebook and then on to cell 5

analysis.create_cluster_summary_figure(
    cell_data, psth_z, sta_results, params, CT_directory, old
)

plt.close("all")

gc.collect()

## Cell 9: When satisfied with the clustering, save data

In [ ]:
# 2022-01-23 RWD: leaving this as a block for now instead of changing it into a function.

exp = params.exp
fsave = os.path.join(CT_directory, "{}_cell_typing_data".format(exp))
utils.save_obj(cell_data, fsave)

# fsave = os.path.join(output_directory, 'Exp{}_clustermodel'.format(exp) )
# save_obj([model,psth_z],fsave)

# ----------------------work in progress-----------------------------------

# 2026-01-22 RWD: not converting anything below until get confirmation about what is needed or not.

In [ ]:
def calcium_exp(x):
    a = 0
    b = 1.086
    c = 1.747 - 1.0  ##################### changes here
    val = a + b * np.exp(-c * x)
    return val


def toCalciumLinear(time_sequence, spike_train):
    calcium_filter = calcium_exp(time_sequence)

    stitch3 = np.append(spike_train, [spike_train, spike_train])

    calcium_trace = np.convolve(stitch3, calcium_filter, "full")[
        len(spike_train) : len(spike_train) * 2
    ]
    calcium_trace = calcium_trace - min(calcium_trace)
    calcium_trace = calcium_trace / max(calcium_trace)
    return calcium_trace

# Matching with Baden et al. clusters

In [ ]:
types_matching_folder = "./types_matching"
baden = loadmat(os.path.join(types_matching_folder, "baden_data.mat"))
calcium = loadmat(os.path.join(types_matching_folder, "calcium_conversion.mat"))

# the normalized profile of the chirp stimulus used in Baden et al.  #31988 points
chirp_stim = baden["chirp_stim"][:, 0] / max(baden["chirp_stim"][:, 0])

# the triggers used in Baden et al. plus two seconds (to match them with our stimulus)  #31988 points
chirp_stim_time = baden["chirp_stim_time"][0, :] + 2

# a 249 points array from 0 to 32 seconds. Times of calcium sampling at 8Hz?
baden_time_original = baden["chirp_time"][0, :]

# a (11210, 1) array. I suppose 11210 were the measurements taken and in this array are the labels of the groups
# in which each measurment was clustered
group_idx = baden["group_idx"]

# the average calcium traces for each measurement. Each measurement has 249 time points.
psth_euler = baden["chirp_avg"]

# here I load from a csv the labels of the Baden types
euler_labels_f = open(os.path.join(types_matching_folder, "Baden Types"))
euler_labels_f = csv.reader(euler_labels_f, delimiter=",")
euler_labels = {}  # this contains the 32 labels of the euler cell types
c = 0
for row in euler_labels_f:
    if c == 0:
        stim_cond_head = row
        c = 1
    else:
        euler_labels[c - 1] = row
        c += 1
# -------------------

n_baden_types = len(euler_labels)

# our stimulus. Load it and normalize it
vec_path = "./types_matching/Euler_50Hz_20reps_1024x768pix.vec"
euler_vec = np.genfromtxt(vec_path)
euler_vec = euler_vec[151 : 151 + 1600, 1] / max(euler_vec[151 : 151 + 1600, 1])

#### Repeat this check for the older chirp version!!!!!!

nb_chirp_bins = cell_data[cells[0]]["psth"].size

plt.figure(figsize=(16, 4))
time_stim = np.linspace(0, 32, 1600 + 1)[:-1]
time = np.linspace(0, 32, nb_chirp_bins + 1)[:-1]
# Our chirp
plt.plot(time_stim, euler_vec, label="our chirp")
euler_vec.shape
euler_vec[::5].shape, time.shape
# Baden chirp
plt.plot(chirp_stim_time, chirp_stim + 1.5, label="Baden chirp")
chirp_stim_time.shape, chirp_stim.shape
plt.legend(frameon=False)

# Comparison range:
comp_range = [0.1, 31.9]  # to avoid prestep effects ### why 0.1 and not 2 ?
delta_range = comp_range[1] - comp_range[0]
time_common = np.linspace(comp_range[0], comp_range[1], int(delta_range / 0.05) + 2)

In order to make ours and the baden traces match we have to shift Baden's sampling times of two seconds. But then this will create missing sampling times for Baden and so here we take the last two seconds worth of Baden's sampling times and we stitch them at the beginning of the sampling times sequence

In [ ]:
# the two additional seconds at the end of Baden stim that will be stitched in the beginning
baden_time = baden_time_original + 2
baden_first = baden_time[-15:] - 32
baden_time = np.append(baden_first, baden_time[:-15])

# we do the same for the calcium traces
Baden_types = []

plt.figure(figsize=(8, 16))

for i in np.arange(n_baden_types) + 1:
    # here the calcium traces of all the recordings of each type are averaged together
    trace = np.mean(psth_euler[:, (group_idx == i)[:, 0]], 1)
    # and here they are all put between 0 and 1
    trace = trace - min(trace)
    trace = trace / max(trace)
    trace = np.append(
        trace[-15:], trace[:-15]
    )  # I stitch the last 2 s in the beggining so that it is
    #  the same as my experimental stim
    Baden_types.append(trace)
    plt.plot(baden_time, trace - i)
    plt.text(35, -i, str(i) + "  " + euler_labels[i - 1][0])

## Transform the clustering results into calcium traces

In [ ]:
## make it loadable!!

labels = model.labels_
exp_labels = np.sort(np.unique(labels))
psth_z.shape, "Ncells  - Ndatapoints"

In [ ]:
# Generate experiment type traces
Exp_types = []
fig = plt.figure(figsize=(12, 24))
fig.add_subplot(1, 2, 1)
for i in exp_labels:
    # I repeat to my spiking data the treatment I did for the Euler's calcium traces
    trace = np.mean(psth_z[(labels == i), :], 0)
    trace = trace - min(trace)
    trace = trace / max(trace)
    Exp_types.append(trace)
    plt.plot(time, trace + i)
    plt.title("Mean psth")

# Experiment to calcium
Exp_types_Ca = {}
fig.add_subplot(1, 2, 2)
for i in exp_labels:
    # I transform each cluster's mean psth into a calcium trace
    trace = toCalciumLinear(time, Exp_types[i])
    Exp_types_Ca[i] = (
        trace  # the only difference is that my made-up calcium traces have the dimension of the
    )
    # number of bins the PSTHs had while the baden traces have dimension 249
    plt.plot(time, trace + i)
    plt.title("Mean Calcium transform")

# Correlate Groups

In [ ]:
# Interpolate both calcium traces (Baden and transformed traces) and then calculate them on the same set of times
Baden_common = {}
for i in np.arange(n_baden_types):
    f = interpolate.interp1d(baden_time, Baden_types[i])
    Baden_common[i] = f(
        time_common
    )  # use interpolation function returned by `interp1d`

Exp_common = {}
for i in exp_labels:
    f = interpolate.interp1d(time, Exp_types_Ca[i])
    Exp_common[i] = f(time_common)  # use interpolation function returned by `interp1d`

# n_matches = 29
corr_table = np.zeros([len(exp_labels), n_baden_types])
corr_match = np.zeros([len(exp_labels), n_baden_types])
corr_match_vals = np.zeros([len(exp_labels), n_baden_types])
delta_match = np.zeros([len(exp_labels), n_baden_types + 1])

# For each of our generated calcium type we compute the corr coef with each Baden type
for i in exp_labels:
    for j in np.arange(n_baden_types):
        corr_table[i, j] = np.corrcoef(Exp_common[i], Baden_common[j])[0, 1]
    # For each calcium type, sort de corr coefs in descending order (max first)
    corr_match[i, :] = np.flip(np.argsort(corr_table[i, :])[-32:])  # sorted indices
    corr_match_vals[i, :] = corr_table[i, :][
        np.flip([np.argsort(corr_table[i, :])[-32:]])
    ]  # sorted corr coef values
    # Select the 10 larger corr coefs and store the difference with the next one
    for j in np.arange(10):
        # each row is one of our clusters and each column is the corr.coeff of that cluster with a Baden one. Decreasing order
        delta_match[i, j] = corr_match_vals[i, j] - corr_match_vals[i, j + 1]

# Manual selection of groups

In [ ]:
my_clusters = range(68)
my_clusters = [0, 4, 5, 8, 18, 24, 32, 50, 54]

colors = [
    "orange",
    "r",
    "g",
    "pink",
    "y",
    "gray",
    "k",
    "c",
    "m",
    "darkgray",
    "coral",
    "gold",
    "plum",
    "wheat",
    "navy",
]
#             0     1   2    3     4    5     6   7   8      9         10      11     12      13     14

baden_cluster = 23

for my_clus in my_clusters:
    i = np.where(corr_match[my_clus, :] == baden_cluster)[0][0]
    plt.figure(figsize=(12, 2))
    plt.plot(baden_time, Baden_types[baden_cluster], lw=3)
    plt.title("{}, {}".format(euler_labels[baden_cluster], my_clus))

    plt.plot(time, Exp_types_Ca[my_clus], color=colors[my_clus % 15])
#     plt.xlabel('match '+colors[n_match%15]+' n ' +str(n_match)+' val '+str(np.round(corr_match_vals[cluster_idx][n_match]*100)) )
#     print('match',str(n_match),' ',j, 'Baden ',i)
# ----------------------------------------------------
# manual_selection = corr_match[:,0].astype('int')

# manual_selection=np.zeros()

# manual_selection[5] = corr_match[5,5]
# manual_selection[6] = corr_match[6,14]
# manual_selection[7] = corr_match[7,9]
# manual_selection[9] = corr_match[9,3]
# manual_selection[10] = corr_match[10,21]
# manual_selection[13] = corr_match[13,12]
# manual_selection[14] = corr_match[14,15]
# manual_selection[15] = corr_match[15,15]
# manual_selection[18] = corr_match[18,4]
# manual_selection[19] = corr_match[19,9]
# manual_selection[20] = corr_match[20,4]
# manual_selection[22] = corr_match[22,12]
# manual_selection[25] = corr_match[25,2]
# manual_selection[27] = corr_match[27,10]
# manual_selection[29] = corr_match[29,1]
# manual_selection[32] = corr_match[32,10]

# manual_selection[37] = corr_match[37,25]
# manual_selection[38] = corr_match[38,13]
# manual_selection[41] = corr_match[41,18]
# manual_selection[46] = corr_match[46,1]
# manual_selection[64] = corr_match[64,13]
# manual_selection[66] = corr_match[66,3]
# manual_selection[67] = corr_match[67,21]
# manual_selection[68] = corr_match[68,16]
# manual_selection[69] = corr_match[69,5]
# manual_selection[70] = corr_match[70,1]
# manual_selection[72] = corr_match[72,16]
# manual_selection[73] = corr_match[73,18]

# manual_selection[74] = corr_match[74,3]

# manual_selection[34] = corr_match[34,1]
# manual_selection[62] = corr_match[62,1]
# manual_selection[42] = corr_match[42,1]

# manual_selection[21] =corr_match[21,2]

# manual_selection[48] =corr_match[48,0]

# #----------------------------------------
# man_vals= np.zeros(len(exp_labels))
# for i in exp_labels:
#     if manual_selection[i]!=-1:
#         match = np.where(manual_selection[i]==corr_match[i,:])[0][0]
#         man_vals[i] = corr_match_vals[i,match]

In [ ]:
# code to replot the cell group ID after the matching

In [ ]:
# code to save the Euler label after the matching

# SAVE DATA

In [ ]:
fsave = os.path.join(params.output_directory, "{}_cell_typing_data".format(exp))
utils.save_obj(cell_data, fsave)

# fsave = os.path.join(output_directory, 'Exp{}_clustermodel'.format(exp) )
# save_obj([model,psth_z],fsave)

# fsave = os.path.join(output_directory, 'Exp{}_selected_cells'.format(exp) )
# save_obj(selected_cells,fsave)

### Old code

In [ ]:
# get cell templates

# template_path = params.phy_directory

# spike_templates = np.load(os.path.join(template_path, 'spike_templates.npy'))   #one template ref per spike
# templates = np.load(os.path.join(template_path,"templates.npy"))
# spike_clusters = np.load(os.path.join(template_path, 'spike_clusters.npy'))

# plt.figure()
# for cell_nb in (good_clusters[:]):   # Is the correct template recovered ?
#     temp_inds=np.unique(spike_templates[spike_clusters==int(cell_nb)])
#     # per each cell temp_inds is a list of the different templates that were assigned to this cell throughout the recording (sometimes it might not be a single one)

#     #----------------

#     maxt=0
#     for t in temp_inds:
#         for nt in range(templates.shape[2]):
#             if maxt<abs(np.min(templates[t,:,nt])):
#                 maxt=abs(np.min(templates[t,:,nt]))
#                 el_sel=nt
#                 temp_sel=t
#                 # here I select the template index with the biggest spike amplitude. But each template index
#                 # has several corresponding templates (why?). So even among those, given one template index, I
#                 # chose the one with the biggest spike amplitude

# #     plt.title(cell_nb)
#     plt.plot(templates[temp_sel,:,el_sel])
#     cell_data[cell_nb]["template"] = templates[temp_sel,:,el_sel]